In [ ]:
# Python implementation for calculating duration

import numpy as np



# Convert raw audio bytes to array of sample values

samples = np.frombuffer(audio_data, dtype=np.int16)



# Get the count of samples

num_samples = len(samples)  # e.g., 48,000



# Define the sample rate (samples per second)

sample_rate = 16000



# Calculate duration in seconds

duration = num_samples / sample_rate  # 48,000 / 16,000 = 3.0



print(f"Recording duration: {duration:.2f} seconds")

In [ ]:
import numpy as np

# Example: audio_data should be raw bytes (make sure it's not empty)
# audio_data = b'...'

# Convert raw audio bytes to array of sample values
samples = np.frombuffer(audio_data, dtype=np.int16)

# Check if audio_data is valid
if samples.size == 0:
    print("No audio data found.")
else:
    # Get the count of samples
    num_samples = len(samples)

    # Define the sample rate (samples per second)
    sample_rate = 16000  # Change if your audio uses a different rate

    # Calculate duration in seconds
    duration = num_samples / sample_rate

    print(f"Recording duration: {duration:.2f} seconds")

In [ ]:
import numpy as np
import wave

with wave.open("audio.wav", "rb") as wf:
    audio_data = wf.readframes(wf.getnframes())
    sample_rate = wf.getframerate()

samples = np.frombuffer(audio_data, dtype=np.int16)

num_samples = len(samples)
duration = num_samples / sample_rate

print(f"Recording duration: {duration:.2f} seconds")

In [ ]:
"""

L2: Voice Analysis Lab

Record → Analyze → Compare Two Recordings



============== DEPENDENCY SETUP ==============

CHECK IF INSTALLED: pip show SpeechRecognition pyaudio numpy matplotlib



INSTALL - WINDOWS:

pip install SpeechRecognition pyaudio numpy matplotlib



INSTALL - macOS:

brew install portaudio

pip install SpeechRecognition pyaudio numpy matplotlib

==============================================

"""



"""
L1: Basic Speech-to-Text
Record → Save → Transcribe → Visualize Waveform
"""
import threading
import sys
import time
import pyaudio
import numpy as np
import matplotlib.pyplot as plt
import wave
import speech_recognition as sr
from speech_recognition import AudioData

stop_event = threading.Event()

def wait_for_enter():
    input("\n🎤 Press Enter to stop recording...\n")
    stop_event.set()

def spinner():
    chars = '|/-\\'
    i = 0
    while not stop_event.is_set():
        sys.stdout.write(f'\r🔴 Recording... {chars[i % 4]}')
        sys.stdout.flush()
        i += 1
        time.sleep(0.1)
    print("\r✅ Recording complete!          ")

def record_audio():
    p = pyaudio.PyAudio()
    stream = p.open(format=pyaudio.paInt16, channels=1, rate=16000,
                    input=True, frames_per_buffer=1024)
    frames = []

    threading.Thread(target=wait_for_enter, daemon=True).start()
    threading.Thread(target=spinner, daemon=True).start()

    while not stop_event.is_set():
        frames.append(stream.read(1024))

    stream.stop_stream()
    stream.close()
    width = p.get_sample_size(pyaudio.paInt16)
    p.terminate()
    return b''.join(frames), 16000, width

def save_audio(data, rate, width, filename="recording.wav"):
    with wave.open(filename, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(width)
        wf.setframerate(rate)
        wf.writeframes(data)
    print(f"💾 Saved: {filename}")

def transcribe(data, rate, width):
    recognizer = sr.Recognizer()
    audio = AudioData(data, rate, width)
    try:
        text = recognizer.recognize_google(audio)
        print(f"📝 Transcription: {text}")
    except sr.UnknownValueError:
        print("❌ Could not understand audio")
    except sr.RequestError as e:
        print(f"❌ API Error: {e}")

def plot_waveform(data, rate):
    samples = np.frombuffer(data, dtype=np.int16)
    time_axis = np.linspace(0, len(samples) / rate, len(samples))
    plt.figure(figsize=(10, 4))
    plt.plot(time_axis, samples, color='blue')
    plt.title("Your Voice Waveform")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

def main():
    print("=" * 40)
    print("🎙️  HELLO AI, CAN YOU HEAR ME?")
    print("=" * 40)
    print("\nSpeak into your microphone...")

    audio_data, rate, width = record_audio()
    save_audio(audio_data, rate, width)
    transcribe(audio_data, rate, width)
    plot_waveform(audio_data, rate)

if __name__ == "__main__":
    main()